# 06 — Embedding similarity (proposal 3.3.6) → RQ2 / RQ3

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Reuse cached SBERT embeddings. Centroids → cosine → Ward dendrogram (RQ3). Domain-wise LLM→human distances (RQ2).

In [ ]:
from src import data, features, analysis, viz
clean = data.load_or_build_clean()
emb = features.build_sbert(clean['text'])   # cached
y = clean[config.LABEL_COL].values

In [ ]:
cents = analysis.class_centroids(emb, y)
sim = analysis.centroid_cosine_matrix(cents)
link = analysis.ward_dendrogram_linkage(cents)
sim

In [ ]:
labels = list(cents.keys())

fig, ax = plt.subplots(figsize=(8, 7), facecolor=viz.SURFACE)
im = ax.imshow(sim.values, cmap=viz.SEQUENTIAL_BLUE, vmin=sim.values[~np.eye(len(labels), dtype=bool)].min(), vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
ax.set_title('SBERT class-centroid cosine similarity')
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Cosine similarity')
fig.tight_layout()
save_fig(fig, 'centroid_cosine_heatmap')

In [ ]:
from scipy.cluster.hierarchy import dendrogram

fig, ax = plt.subplots(figsize=(9, 5), facecolor=viz.SURFACE)
dendrogram(link, labels=labels, ax=ax, color_threshold=0, above_threshold_color=viz.INK_SECONDARY)
ax.set_title('Ward clustering of SBERT class centroids (1 − cosine similarity)\ncompare groupings against config.MODEL_FAMILIES')
ax.set_ylabel('Distance')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelrotation=45, labelsize=8, colors=viz.INK_SECONDARY)
ax.tick_params(axis='y', colors=viz.INK_SECONDARY)
fig.tight_layout()
save_fig(fig, 'centroid_dendrogram')

In [ ]:
dom = analysis.domain_human_distances(emb, clean)
dom   # sorted ascending -- top rows are the LLMs closest to human writing

In [ ]:
# Single-series ranked chart -- one hue (magnitude, not category identity),
# error bars carry each LLM's domain-to-domain variability (RQ2).
dom_sorted = dom.sort_values('mean_distance', ascending=True)

fig, ax = viz.new_fig(figsize=(7, 6))
y_pos = np.arange(len(dom_sorted))
ax.barh(y_pos, dom_sorted['mean_distance'], xerr=dom_sorted['std_distance'],
        color=viz.SEQUENTIAL_BLUE(0.6), ecolor=viz.INK_MUTED, capsize=3, zorder=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(dom_sorted.index)
ax.set_xlabel('Mean cosine distance to within-domain human centroid')
ax.set_title('LLM → human semantic distance, averaged across 8 domains\n(shorter bar = more human-like writing; error bar = domain variability)')
fig.tight_layout()
save_fig(fig, 'domain_human_distance')